<a href="https://colab.research.google.com/github/Susan-sapkota-98/Nepali-Whisper-ASR/blob/main/Nepali_Whisper_ASR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**“How does the amount of fine-tuning data affect Whisper’s error rate on Nepali speech, and does it work on audio from outside the training dataset?”**

In [1]:
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
from IPython.display import Audio

SEED = 42
BASE_URL = "https://openslr.trmal.net/resources/54"

DATA_DIR = Path("/content/data")
ZIP_PATH = DATA_DIR / "asr_nepali_0.zip"
TSV_PATH = DATA_DIR / "utt_spk_text.tsv"
AUDIO_DIR = DATA_DIR / "audio"

DATA_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
if not TSV_PATH.exists():
    !wget -q -O {TSV_PATH} {BASE_URL}/utt_spk_text.tsv
if not ZIP_PATH.exists():
    !wget -q -O {ZIP_PATH} {BASE_URL}/asr_nepali_0.zip

!ls -lh {DATA_DIR}

total 573M
-rw-r--r-- 1 root root 562M Jun 28  2018 asr_nepali_0.zip
-rw-r--r-- 1 root root  11M Jun 29  2018 utt_spk_text.tsv


In [3]:
if not AUDIO_DIR.exists():
    !unzip -q {ZIP_PATH} -d {AUDIO_DIR}

In [4]:
transcripts = pd.read_csv(
    TSV_PATH,
    sep="\t",
    header=None,
    names=["file_id", "speaker_id", "text"],
)
print("rows in tsv:", len(transcripts))

rows in tsv: 157905


In [ ]:
flac_files = list(AUDIO_DIR.rglob("*.flac"))
audio_path_by_id = {path.stem: path for path in flac_files}

dataset = transcripts[transcripts["file_id"].isin(audio_path_by_id)].copy()
dataset["audio_path"] = dataset["file_id"].map(audio_path_by_id).astype(str)
dataset = dataset.reset_index(drop=True)

audio_without_text = set(audio_path_by_id) - set(transcripts["file_id"])

print("flac files:", len(flac_files))
print("rows with audio:", len(dataset))
print("audio files without a transcript:", len(audio_without_text))
print("speakers:", dataset["speaker_id"].nunique())

flac files: 0
rows with audio: 0
audio files without a transcript: 0
speakers: 0


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

print("--- Drive data folder ---")
!ls -lh /content/drive/MyDrive/nepali_whisper_asr/data
print("--- Drive model folders ---")
!ls /content/drive/MyDrive/nepali_whisper_asr
print("--- local files ---")
!ls /content/data
print("--- GPU ---")
!nvidia-smi --query-gpu=name --format=csv

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--- Drive data folder ---
ls: cannot access '/content/drive/MyDrive/nepali_whisper_asr/data': No such file or directory
--- Drive model folders ---
checkpoints  final_model
--- local files ---
asr_nepali_0.zip  utt_spk_text.tsv
--- GPU ---
name
Tesla T4


In [ ]:
sample = dataset.sample(1, random_state=3).iloc[0]
print("text:", sample["text"])

waveform, sample_rate = sf.read(sample["audio_path"])
print("sample rate:", sample_rate, "| duration (s):", round(len(waveform) / sample_rate, 2))

Audio(waveform, rate=sample_rate)

ValueError: a must be greater than 0 unless no samples are taken

In [ ]:
check_subset = dataset.sample(min(500, len(dataset)), random_state=SEED)
durations = [sf.info(path).duration for path in check_subset["audio_path"]]

pd.Series(durations).describe()

,0
count,0
unique,0
top,NaN
freq,NaN


## 4. Train / validation / test split
- Split by speaker, so test speakers are never seen in training (80 / 10 / 10)
- Remove validation and test rows whose text also appears in train (text leakage)
- Audio is already 16 kHz, so no resampling is needed for Whisper

In [ ]:
speaker_ids=dataset["speaker_id"].unique()
# print(len(speaker_ids))

rng=np.random.default_rng(SEED)
rng.shuffle(speaker_ids)

n_eval_speakers=int(0.1*len(speaker_ids))
test_speakers=set(speaker_ids[:n_eval_speakers])
val_speakers=set(speaker_ids[n_eval_speakers:2*n_eval_speakers])

test_df=dataset[dataset["speaker_id"].isin(test_speakers)]
val_df=dataset[dataset["speaker_id"].isin(val_speakers)]
train_df=dataset[~dataset["speaker_id"].isin(test_speakers | val_speakers)]

print("Before removing repeated text -> val:", len(val_df)," | test:",len(test_df))

train_texts=set(train_df["text"])
val_df=val_df[~val_df["text"].isin(train_texts)]
test_df=test_df[~test_df["text"].isin(train_texts)]

print(f"train: {len(train_df)} | val: {len(val_df)} | test: {len(test_df)}")

Before removing repeated text -> val: 0  | test: 0
train: 0 | val: 0 | test: 0


## Leakage Check

In [ ]:
assert not set(train_df["speaker_id"]) & set(test_df["speaker_id"])
assert not set(train_df["speaker_id"]) & set(val_df["speaker_id"])
assert not set(train_df["text"]) & set(test_df["text"])
assert not set(train_df["text"]) & set(val_df["text"])
print("no speaker or text overlap between the splits")

no speaker or text overlap between the splits


In [ ]:
REPO_DIR = Path("/content/Nepali-Whisper-ASR")
split_dir = REPO_DIR / "results" / "splits"
split_dir.mkdir(parents=True, exist_ok=True)

for split_name, split_df in {"train": train_df, "val": val_df, "test": test_df}.items():
    split_df[["file_id", "speaker_id", "text"]].to_csv(split_dir / f"{split_name}.csv", index=False)

!ls -lh {split_dir}

total 12K
-rw-r--r-- 1 root root 24 Nov  5 23:49 test.csv
-rw-r--r-- 1 root root 24 Nov  5 23:49 train.csv
-rw-r--r-- 1 root root 24 Nov  5 23:49 val.csv


In [ ]:
!pip install -q transformers accelerate evaluate jiwer
!nvidia-smi

Thu Nov  5 23:50:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P0             29W /   70W |    1119MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!nvidia-smi

Wed Nov  4 00:16:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# !pip install jiwer

In [ ]:
import torch
import jiwer
from transformers import WhisperForConditionalGeneration, WhisperProcessor

MODEL_NAME = "openai/whisper-small"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

processor = WhisperProcessor.from_pretrained(MODEL_NAME, language="nepali", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)
_ = model.eval()

device: cuda


preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

... (4 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  967MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

In [ ]:
import numpy as np

label_lengths = [
    len(processor.tokenizer(text).input_ids)
    for text in train_df["text"].sample(1000, random_state=SEED)
]
print(pd.Series(label_lengths).describe())

MAX_NEW_TOKENS = int(np.percentile(label_lengths, 99)) + 10
print("MAX_NEW_TOKENS:", MAX_NEW_TOKENS)

ValueError: a must be greater than 0 unless no samples are taken

In [ ]:
def transcribe(audio_paths, batch_size=16, **generate_kwargs):
    predictions = []
    for start in range(0, len(audio_paths), batch_size):
        batch_paths = audio_paths[start:start + batch_size]
        waveforms = [sf.read(path)[0] for path in batch_paths]

        inputs = processor.feature_extractor(
            waveforms, sampling_rate=16000, return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            generated_ids = model.generate(
                inputs.input_features,
                language="nepali",
                task="transcribe",
                **generate_kwargs,
            )
        predictions.extend(processor.batch_decode(generated_ids, skip_special_tokens=True))
    return [prediction.strip() for prediction in predictions]


def score(references, predictions):
    return round(jiwer.wer(references, predictions), 3), round(jiwer.cer(references, predictions), 3)

In [ ]:
eval_df = test_df.sample(100, random_state=SEED).reset_index(drop=True)
references = eval_df["text"].tolist()
audio_paths = eval_df["audio_path"].tolist()
has_digit = eval_df["text"].str.contains("[०-९0-9]").to_numpy()

settings = {
    "raw": dict(max_new_tokens=MAX_NEW_TOKENS),
    "repeat_guard": dict(max_new_tokens=MAX_NEW_TOKENS, no_repeat_ngram_size=4),
}

all_predictions = {}
for name, kwargs in settings.items():
    predictions = transcribe(audio_paths, **kwargs)
    all_predictions[name] = predictions

    wer_all, cer_all = score(references, predictions)
    print(f"[{name}] all 100 clips -> WER {wer_all} | CER {cer_all}")

    for label, mask in [("with digits", has_digit), ("no digits", ~has_digit)]:
        subset_refs = [r for r, keep in zip(references, mask) if keep]
        subset_preds = [p for p, keep in zip(predictions, mask) if keep]
        if subset_refs:
            wer_sub, cer_sub = score(subset_refs, subset_preds)
            print(f"    {label} ({len(subset_refs)} clips) -> WER {wer_sub} | CER {cer_sub}")

for ref, pred in list(zip(references, all_predictions["repeat_guard"]))[:10]:
    print("REF :", ref)
    print("PRED:", pred)
    print()

[raw] all 100 clips -> WER 1.728 | CER 0.785
    with digits (6 clips) -> WER 1.947 | CER 1.36
    no digits (94 clips) -> WER 1.713 | CER 0.749
[repeat_guard] all 100 clips -> WER 1.541 | CER 0.68
    with digits (6 clips) -> WER 2.105 | CER 1.279
    no digits (94 clips) -> WER 1.502 | CER 0.642
REF : जेम्स वन्डको छैठौँ
PRED: जिम्स वान्द को चाइतो

REF : मा सोभियत अन्तरिक्ष
PRED: मा सोगेत अंतरिच्या

REF : जुन इलाहाबादबाट हल्दिया
PRED: जून इला हा बाद बातर हल दिया

REF : सिटी अफ होप
PRED: अपन्ट्टी अफवुप

REF : अनि नम्बर प्लेट
PRED: अनी नमवर प्लेट

REF : जुनमा भएको मिडवेको
PRED: जिन्मा बाए को मिध वे को

REF : त्यहाँ भन्दा लाज
PRED: तेहां बन्दा लाज

REF : झाडग्राम भारतका सहरहरू
PRED: चार्द ग्राम भारद्का सहर अरु

REF : किताब बेचबिखन गर्ने
PRED: अप तो तो में प्रदागा प्च्टाए तो सब तो किता बेच पिखण गरने तो

REF : उनी फेरि भाग्न
PRED: उनी प्री बागना



In [ ]:
%cd /content
!mv /content/Nepali-Whisper-ASR /content/Nepali-Whisper-ASR_old

from google.colab import userdata
token = userdata.get("GITHUB_TOKEN")

!git clone https://{token}@github.com/Susan-sapkota-98/Nepali-Whisper-ASR.git /content/Nepali-Whisper-ASR
%cd /content/Nepali-Whisper-ASR

!git config user.name "Susan Sapkota"
!git config user.email "susansapkota986@gmail.com"

!ls -a

/content
Cloning into '/content/Nepali-Whisper-ASR'...
remote: Enumerating objects: 20, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 20 (delta 3), reused 16 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (20/20), 232.14 KiB | 8.00 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/Nepali-Whisper-ASR
.  ..  .git  .gitignore  notebooks  README.md  results	src


In [ ]:
%%writefile .gitignore
*.zip
*.flac
*.wav
*.mp3
data/
checkpoints/
__pycache__/
.ipynb_checkpoints/

Overwriting .gitignore


In [ ]:
%cd /content/Nepali-Whisper-ASR
!git status --short
!git add .
!git commit -m "Add baseline Whisper-small results and data splits"
!git push

/content/Nepali-Whisper-ASR
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Everything up-to-date


In [ ]:
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

full_references=test_df["text"].tolist()
full_prediction=transcribe(
    test_df["audio_path"].tolist(),
    max_new_tokens=MAX_NEW_TOKENS,
    no_repeat_ngram_size=4,
)
wer_full,cer_full=score(full_references,full_prediction)
print(f"[{name}] all 100 clips -> WER {wer_full} | CER {cer_full}")

[repeat_guard] all 100 clips -> WER 1.363 | CER 0.589


In [ ]:
full_predictions = full_prediction
print("predictions:", len(full_predictions))

predictions: 898


In [ ]:
print("train / val / test:", len(train_df), len(val_df), len(test_df))
print("predictions:", len(full_predictions))

train / val / test: 7961 933 898
predictions: 898


In [ ]:
import json

REPO_DIR = Path("/content/Nepali-Whisper-ASR")
RESULTS_DIR = REPO_DIR / "results"
SPLIT_DIR = RESULTS_DIR / "splits"
assert (REPO_DIR / ".git").exists(), "repo folder missing - clone the repo again first"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

for split_name, split_df in {"train": train_df, "val": val_df, "test": test_df}.items():
    split_df[["file_id", "speaker_id", "text"]].to_csv(SPLIT_DIR / f"{split_name}.csv", index=False)

wer_full, cer_full = score(test_df["text"].tolist(), full_predictions)
print(f"baseline on full test set ({len(test_df)} clips) -> WER {wer_full} | CER {cer_full}")

baseline_df = test_df[["file_id", "speaker_id", "text"]].copy()
baseline_df["prediction"] = full_predictions
baseline_df.to_csv(RESULTS_DIR / "baseline_predictions.csv", index=False)

baseline_metrics = {
    "model": MODEL_NAME,
    "test_clips": len(test_df),
    "settings": {"max_new_tokens": MAX_NEW_TOKENS, "no_repeat_ngram_size": 4},
    "wer": wer_full,
    "cer": cer_full,
}
with open(RESULTS_DIR / "baseline_metrics.json", "w", encoding="utf-8") as f:
    json.dump(baseline_metrics, f, indent=2)

!ls -lh {RESULTS_DIR}

baseline on full test set (898 clips) -> WER 1.363 | CER 0.589
total 120K
-rw-r--r-- 1 root root  167 Nov  4 00:22 baseline_metrics.json
-rw-r--r-- 1 root root 112K Nov  4 00:22 baseline_predictions.csv
drwxr-xr-x 2 root root 4.0K Nov  4 00:18 splits


In [ ]:
%cd /content/Nepali-Whisper-ASR
!git status --short
!git add .
!git commit -m "Add baseline Whisper-small results and data splits"
!git push

/content/Nepali-Whisper-ASR
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Everything up-to-date


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

CHECKPOINT_DIR = Path("/content/drive/MyDrive/nepali_whisper_asr/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print("checkpoints will be saved to:", CHECKPOINT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
checkpoints will be saved to: /content/drive/MyDrive/nepali_whisper_asr/checkpoints


In [5]:
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 45.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [jiwer]


In [6]:
# ===== SETUP CELL: run this first every time you reopen the notebook =====
import json
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import jiwer
from google.colab import drive, userdata
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()

SEED = 42
MODEL_NAME = "openai/whisper-small"
MAX_NEW_TOKENS = 73

drive.mount("/content/drive")
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/nepali_whisper_asr/data")
CHECKPOINT_DIR = Path("/content/drive/MyDrive/nepali_whisper_asr/checkpoints")
DATA_DIR = Path("/content/data")
AUDIO_DIR = DATA_DIR / "audio"
REPO_DIR = Path("/content/Nepali-Whisper-ASR")

# 1. data: copy the zip from Drive, then unzip
DATA_DIR.mkdir(parents=True, exist_ok=True)
if not (DATA_DIR / "asr_nepali_0.zip").exists():
    !cp {DRIVE_DATA_DIR}/asr_nepali_0.zip {DATA_DIR}/
if not AUDIO_DIR.exists():
    !unzip -q {DATA_DIR}/asr_nepali_0.zip -d {AUDIO_DIR}

# 2. repo: clone only if it is missing
if not REPO_DIR.exists():
    token = userdata.get("GITHUB_TOKEN")
    !git clone https://{token}@github.com/Susan-sapkota-98/Nepali-Whisper-ASR.git {REPO_DIR}
!git -C {REPO_DIR} config user.name "Susan Sapkota"
!git -C {REPO_DIR} config user.email "YOUR_GITHUB_EMAIL"

# 3. splits: read the saved csv files, so the split is always the same
audio_path_by_id = {path.stem: str(path) for path in AUDIO_DIR.rglob("*.flac")}
splits = {}
for split_name in ["train", "val", "test"]:
    split_df = pd.read_csv(REPO_DIR / "results" / "splits" / f"{split_name}.csv")
    split_df["audio_path"] = split_df["file_id"].map(audio_path_by_id)
    assert split_df["audio_path"].notna().all(), f"missing audio in {split_name}"
    splits[split_name] = split_df
train_df, val_df, test_df = splits["train"], splits["val"], splits["test"]
print("train / val / test:", len(train_df), len(val_df), len(test_df))

# 4. model
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
processor = WhisperProcessor.from_pretrained(MODEL_NAME, language="nepali", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)
_ = model.eval()

Mounted at /content/drive
Cloning into '/content/Nepali-Whisper-ASR'...
remote: Enumerating objects: 38, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 38 (delta 9), reused 29 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (38/38), 362.56 KiB | 1.58 MiB/s, done.
Resolving deltas: 100% (9/9), done.
train / val / test: 0 0 0
device: cuda


preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

... (4 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  967MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

## 5. Prepare data for fine-tuning
- `WhisperAsrDataset` reads one audio file and its text, and makes input features + label tokens
- `DataCollatorSpeechSeq2Seq` joins several clips into one batch (padding the short ones)
- Padding in the labels is set to -100 so the loss ignores it

In [ ]:
from dataclasses import dataclass
from typing import Any

import torch
from torch.utils.data import Dataset

#the tokenizer now adds the Nepali + transcribe tokens in front of every label
processor.tokenizer.set_prefix_tokens(language="nepali",task="transcribe")

class WhisperAsrDataset(Dataset):
  def __init__(self,dataframe,processor):
    self.audio_paths = dataframe["audio_path"].tolist()
    self.text = dataframe["text"].tolist()
    self.processor = processor

  def __len__(self):
    return len(self.text)

  def __getitem__(self, index):
      waveform,sample_rate=sf.read(self.audio_paths[index])
      assert sample_rate == 16000, f"unexpected sample rate:{sample_rate}"

      input_features = self.processor.feature_extractor(waveform,sampling_rate=16000).input_features[0]
      labels=self.processor.tokenizer(self.text[index]).input_ids

      return {"input_features": input_features, "labels":labels}


@dataclass
class DataCollatorSpeechSeq2Seq:
    processor:Any
    decoder_start_token_id: int

    def __call__(self,features):
      #audio part: pad to the same size and make tensors
      audio_part=[{"input_features":item["input_features"]} for item in features]
      batch= self.processor.feature_extractor.pad(audio_part,return_tensors="pt")

      #text part: pad to the same length, padding becomes -100 (ignored by the loss)
      text_part=[{"input_ids":item["labels"]} for item in features]
      padded_text = self.processor.tokenizer.pad(text_part, return_tensors="pt")
      labels = padded_text["input_ids"].masked_fill(padded_text["attention_mask"].ne(1), -100)

      # the model adds the start token by itself , so remore it from the labels
      if (labels[:, 0]==self.decoder_start_token_id).all().cpu().item():
        labels = labels[:, 1:]

      batch["labels"]=labels
      return batch

train_dataset= WhisperAsrDataset(train_df,processor)
val_dataset=WhisperAsrDataset(val_df,processor)
data_collator=DataCollatorSpeechSeq2Seq(processor=processor, decoder_start_token_id=model.config.decoder_start_token_id,)

print("train clips: ", len(train_dataset), "| val clips:",len(val_dataset))



####Testing one batch
we test on 4 clips before any training. This catches mistakes in seconds

In [ ]:
from torch.utils.data import DataLoader

test_loader= DataLoader(train_dataset, batch_size=4, shuffle=False, collate_fn=data_collator)
batch=next(iter(test_loader))

print("input_features shape:", tuple(batch["input_features"].shape))
print("labels shape:",tuple(batch["labels"].shape))

# decoded the labels back to text and compare with the original text
labels= batch["labels"].clone()
labels[labels == -100] = processor . tokenizer.pad_token_id

for i in range(4):
  decoded_text=processor.tokenizer.decode(labels[i],skip_special_tokens=True)
  print("Original:",train_df["text"].iloc[i])
  print("Decoded",decoded_text)
  print()
print("labels with speacial tokens:")
print(processor.tokenizer.decode(labels[0], skip_special_tokens=False))

input_features shape: (4, 80, 3000)
labels shape: (4, 28)
Original: ०४६ सालको जनआन्दोलन
Decoded ०४६ सालको जनआन्दोलन

Original: १०० खेल खेल्ने
Decoded १०० खेल खेल्ने

Original: १० १५ दिन खानाले पनि
Decoded १० १५ दिन खानाले पनि

Original: १०५ लगातार कट्स
Decoded १०५ लगातार कट्स

labels with speacial tokens:
<|ne|><|transcribe|><|notimestamps|>०४६ सालको जनआन्दोलन<|endoftext|><|endoftext|>


###***Important Note
#### A small training test

Before the real training, which takes long, wee run only 10 steps at first to check two things: does training start, and does the GPU fit? If it fails, we find out in a minuate, not after an hour

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

model.config.use_cache = False # needed when gradient checkpointing is on

smoke_args= Seq2SeqTrainingArguments(
    output_dir="/content/smoke_test",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    max_steps=10,
    fp16=True,
    gradient_checkpointing=True,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=2,
)

smoke_trainer=Seq2SeqTrainer(
    model=model,
    args=smoke_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)
torch.cuda.reset_peak_memory_stats()
smoke_trainer.train()
print("peak GPU Memory (GB):", round(torch.cuda.max_memory_allocated()/1024**3,2))

{'loss': '3.511', 'grad_norm': 'inf', 'learning_rate': '1e-05', 'epoch': '0.002008'}
{'loss': '3.193', 'grad_norm': 'inf', 'learning_rate': '9e-06', 'epoch': '0.004016'}
{'loss': '3.648', 'grad_norm': 'inf', 'learning_rate': '8e-06', 'epoch': '0.006024'}
{'loss': '3.475', 'grad_norm': '62.73', 'learning_rate': '7e-06', 'epoch': '0.008032'}
{'loss': '2.811', 'grad_norm': '48.11', 'learning_rate': '6e-06', 'epoch': '0.01004'}
{'loss': '2.484', 'grad_norm': '42.26', 'learning_rate': '5e-06', 'epoch': '0.01205'}
{'loss': '2.415', 'grad_norm': '44.07', 'learning_rate': '4e-06', 'epoch': '0.01406'}
{'loss': '2.519', 'grad_norm': '43.55', 'learning_rate': '3e-06', 'epoch': '0.01606'}
{'loss': '2.118', 'grad_norm': '37.74', 'learning_rate': '2e-06', 'epoch': '0.01807'}
{'loss': '2.568', 'grad_norm': '36.34', 'learning_rate': '1e-06', 'epoch': '0.02008'}
{'train_runtime': '28.97', 'train_samples_per_second': '5.522', 'train_steps_per_second': '0.345', 'train_loss': '2.874', 'epoch': '0.02008'}


In [ ]:
import gc

from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

# 1. clean up the smoke test
del smoke_trainer
gc.collect()
torch.cuda.empty_cache()

# 2. fresh model (the smoke test changed the old one)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)
model.config.use_cache = False
model.generation_config.language = "nepali"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None


# 3. metric used during evaluation
def compute_metrics(eval_prediction):
    pred_ids = np.where(
        eval_prediction.predictions != -100,
        eval_prediction.predictions,
        processor.tokenizer.pad_token_id,
    )
    label_ids = np.where(
        eval_prediction.label_ids != -100,
        eval_prediction.label_ids,
        processor.tokenizer.pad_token_id,
    )
    pred_text = [t.strip() for t in processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)]
    label_text = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return {"wer": jiwer.wer(label_text, pred_text), "cer": jiwer.cer(label_text, pred_text)}


# 4. a small random part of validation (random, because the files are sorted by text)
val_small = WhisperAsrDataset(val_df.sample(300, random_state=SEED), processor)

training_args = Seq2SeqTrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=50,
    num_train_epochs=3,
    fp16=True,
    gradient_checkpointing=True,
    eval_strategy="steps",
    eval_steps=250,
    save_strategy="steps",
    save_steps=250,
    save_total_limit=2,
    logging_steps=25,
    predict_with_generate=True,
    generation_max_length=80,
    per_device_eval_batch_size=16,
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=2,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_small,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# 5. plumbing test: evaluate the untrained model on 32 clips
quick_eval_dataset = WhisperAsrDataset(val_df.sample(32, random_state=SEED), processor)
print(trainer.evaluate(eval_dataset=quick_eval_dataset))

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

{'eval_loss': '1.897', 'eval_model_preparation_time': '0.0114', 'eval_wer': '1.091', 'eval_cer': '0.5019', 'eval_runtime': '10.82', 'eval_samples_per_second': '2.958', 'eval_steps_per_second': '0.185', 'epoch': 0}
{'eval_loss': 1.8969212770462036, 'eval_model_preparation_time': 0.0114, 'eval_wer': 1.0909090909090908, 'eval_cer': 0.5019455252918288, 'eval_runtime': 10.8181, 'eval_samples_per_second': 2.958, 'eval_steps_per_second': 0.185, 'epoch': 0}


In [ ]:
has_checkpoint = any(CHECKPOINT_DIR.glob("checkpoint-*"))
trainer.train(resume_from_checkpoint=has_checkpoint)

{'loss': '2.767', 'grad_norm': '25.26', 'learning_rate': '4.8e-06', 'epoch': '0.0502'}
{'loss': '1.439', 'grad_norm': '23.37', 'learning_rate': '9.8e-06', 'epoch': '0.1004'}
{'loss': '1.083', 'grad_norm': '20.5', 'learning_rate': '9.834e-06', 'epoch': '0.1506'}
{'loss': '0.9362', 'grad_norm': '16.99', 'learning_rate': '9.661e-06', 'epoch': '0.2008'}
{'loss': '0.9512', 'grad_norm': '17.71', 'learning_rate': '9.488e-06', 'epoch': '0.251'}
{'loss': '0.7805', 'grad_norm': '19.65', 'learning_rate': '9.314e-06', 'epoch': '0.3012'}
{'loss': '0.7614', 'grad_norm': '19.53', 'learning_rate': '9.141e-06', 'epoch': '0.3514'}
{'loss': '0.71', 'grad_norm': '14.87', 'learning_rate': '8.968e-06', 'epoch': '0.4016'}
{'loss': '0.7018', 'grad_norm': '16.27', 'learning_rate': '8.795e-06', 'epoch': '0.4518'}
{'loss': '0.6603', 'grad_norm': '14.61', 'learning_rate': '8.622e-06', 'epoch': '0.502'}
{'eval_loss': '0.3278', 'eval_model_preparation_time': '0.0114', 'eval_wer': '0.5835', 'eval_cer': '0.1771', 'ev

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.6528', 'grad_norm': '12.84', 'learning_rate': '8.449e-06', 'epoch': '0.5522'}
{'loss': '0.607', 'grad_norm': '12.07', 'learning_rate': '8.276e-06', 'epoch': '0.6024'}
{'loss': '0.6207', 'grad_norm': '13.4', 'learning_rate': '8.102e-06', 'epoch': '0.6526'}
{'loss': '0.6237', 'grad_norm': '17.26', 'learning_rate': '7.929e-06', 'epoch': '0.7028'}
{'loss': '0.578', 'grad_norm': '15.14', 'learning_rate': '7.756e-06', 'epoch': '0.753'}
{'loss': '0.5831', 'grad_norm': '12.34', 'learning_rate': '7.583e-06', 'epoch': '0.8032'}
{'loss': '0.5523', 'grad_norm': '15.83', 'learning_rate': '7.41e-06', 'epoch': '0.8534'}
{'loss': '0.5457', 'grad_norm': '15.47', 'learning_rate': '7.237e-06', 'epoch': '0.9036'}
{'loss': '0.5052', 'grad_norm': '13.75', 'learning_rate': '7.064e-06', 'epoch': '0.9538'}
{'loss': '0.4989', 'grad_norm': '8.66', 'learning_rate': '6.891e-06', 'epoch': '1.004'}
{'eval_loss': '0.2584', 'eval_model_preparation_time': '0.0114', 'eval_wer': '0.4897', 'eval_cer': '0.1369'

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.3372', 'grad_norm': '19.52', 'learning_rate': '6.717e-06', 'epoch': '1.054'}
{'loss': '0.3151', 'grad_norm': '9.477', 'learning_rate': '6.544e-06', 'epoch': '1.104'}
{'loss': '0.323', 'grad_norm': '13.99', 'learning_rate': '6.371e-06', 'epoch': '1.155'}
{'loss': '0.3211', 'grad_norm': '11.98', 'learning_rate': '6.198e-06', 'epoch': '1.205'}
{'loss': '0.3101', 'grad_norm': '14.4', 'learning_rate': '6.025e-06', 'epoch': '1.255'}
{'loss': '0.3085', 'grad_norm': '15.89', 'learning_rate': '5.852e-06', 'epoch': '1.305'}
{'loss': '0.3329', 'grad_norm': '14.06', 'learning_rate': '5.679e-06', 'epoch': '1.355'}
{'loss': '0.2857', 'grad_norm': '10.34', 'learning_rate': '5.506e-06', 'epoch': '1.406'}
{'loss': '0.308', 'grad_norm': '11.78', 'learning_rate': '5.332e-06', 'epoch': '1.456'}
{'loss': '0.3144', 'grad_norm': '14.85', 'learning_rate': '5.159e-06', 'epoch': '1.506'}
{'eval_loss': '0.2456', 'eval_model_preparation_time': '0.0114', 'eval_wer': '0.4634', 'eval_cer': '0.1283', 'eva

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (4 outputs left out)

{'loss': '0.1433', 'grad_norm': '9.059', 'learning_rate': '1.524e-06', 'epoch': '2.56'}
{'loss': '0.1485', 'grad_norm': '10.87', 'learning_rate': '1.35e-06', 'epoch': '2.61'}
{'loss': '0.1354', 'grad_norm': '8.555', 'learning_rate': '1.177e-06', 'epoch': '2.661'}
{'loss': '0.1393', 'grad_norm': '6.6', 'learning_rate': '1.004e-06', 'epoch': '2.711'}
{'loss': '0.1347', 'grad_norm': '10.99', 'learning_rate': '8.31e-07', 'epoch': '2.761'}
{'loss': '0.1201', 'grad_norm': '10.45', 'learning_rate': '6.579e-07', 'epoch': '2.811'}
{'loss': '0.1366', 'grad_norm': '9.765', 'learning_rate': '4.848e-07', 'epoch': '2.861'}
{'loss': '0.12', 'grad_norm': '9.458', 'learning_rate': '3.116e-07', 'epoch': '2.912'}
{'loss': '0.1428', 'grad_norm': '8.817', 'learning_rate': '1.385e-07', 'epoch': '2.962'}
{'eval_loss': '0.2237', 'eval_model_preparation_time': '0.0114', 'eval_wer': '0.4005', 'eval_cer': '0.1091', 'eval_runtime': '68.36', 'eval_samples_per_second': '4.388', 'eval_steps_per_second': '0.278', 'ep

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '4292', 'train_samples_per_second': '5.565', 'train_steps_per_second': '0.348', 'train_loss': '0.4249', 'epoch': '3'}


TrainOutput(global_step=1494, training_loss=0.42485380076979057, metrics={'train_runtime': 4291.8472, 'train_samples_per_second': 5.565, 'train_steps_per_second': 0.348, 'train_loss': 0.42485380076979057, 'epoch': 3.0})

In [ ]:
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()

model.config.use_cache = True
model.eval()


def transcribe(audio_paths, batch_size=16, **generate_kwargs):
    predictions = []
    for start in range(0, len(audio_paths), batch_size):
        batch_paths = audio_paths[start:start + batch_size]
        waveforms = [sf.read(path)[0] for path in batch_paths]

        inputs = processor.feature_extractor(
            waveforms, sampling_rate=16000, return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            generated_ids = model.generate(
                inputs.input_features,
                language="nepali",
                task="transcribe",
                **generate_kwargs,
            )
        predictions.extend(processor.batch_decode(generated_ids, skip_special_tokens=True))
    return [prediction.strip() for prediction in predictions]


def score(references, predictions):
    return round(jiwer.wer(references, predictions), 3), round(jiwer.cer(references, predictions), 3)


test_references = test_df["text"].tolist()
test_audio_paths = test_df["audio_path"].tolist()

after_results = {}
settings = {
    "same_settings_as_baseline": dict(max_new_tokens=MAX_NEW_TOKENS, no_repeat_ngram_size=4),
    "no_repeat_guard": dict(max_new_tokens=MAX_NEW_TOKENS),
}
for name, generate_kwargs in settings.items():
    predictions = transcribe(test_audio_paths, **generate_kwargs)
    wer_value, cer_value = score(test_references, predictions)
    after_results[name] = {"predictions": predictions, "wer": wer_value, "cer": cer_value}
    print(f"[{name}] test set ({len(test_df)} clips) -> WER {wer_value} | CER {cer_value}")

# keep the final model safe on Drive
FINAL_MODEL_DIR = CHECKPOINT_DIR.parent / "final_model"
model.save_pretrained(FINAL_MODEL_DIR)
processor.save_pretrained(FINAL_MODEL_DIR)
!ls -lh {FINAL_MODEL_DIR}

[same_settings_as_baseline] test set (898 clips) -> WER 0.391 | CER 0.106
[no_repeat_guard] test set (898 clips) -> WER 0.382 | CER 0.104


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

total 926M
-rw------- 1 root root 2.2K Nov  4 01:59 config.json
-rw------- 1 root root 4.5K Nov  4 01:59 generation_config.json
-rw------- 1 root root 923M Nov  4 01:59 model.safetensors
-rw------- 1 root root  409 Nov  4 01:59 processor_config.json
-rw------- 1 root root 2.1K Nov  4 01:59 tokenizer_config.json
-rw------- 1 root root 3.8M Nov  4 01:59 tokenizer.json


In [ ]:
last_checkpoint = sorted(CHECKPOINT_DIR.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[1]))[-1]
print("loading:", last_checkpoint.name)
model = WhisperForConditionalGeneration.from_pretrained(last_checkpoint).to(device)

loading: checkpoint-1494


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

In [ ]:
import json

RESULTS_DIR = REPO_DIR / "results"
assert (REPO_DIR / ".git").exists(), "repo folder missing - run the setup cell first"

after_df = test_df[["file_id", "speaker_id", "text"]].copy()
after_df["prediction"] = after_results["same_settings_as_baseline"]["predictions"]
after_df["prediction_no_guard"] = after_results["no_repeat_guard"]["predictions"]
after_df.to_csv(RESULTS_DIR / "finetuned_predictions.csv", index=False)

finetuned_metrics = {
    "model": "whisper-small fine-tuned on SLR54 shard 0",
    "train_clips": len(train_df),
    "epochs": 3,
    "learning_rate": 1e-5,
    "effective_batch_size": 16,
    "test_clips": len(test_df),
    "with_repeat_guard": {k: after_results["same_settings_as_baseline"][k] for k in ("wer", "cer")},
    "without_repeat_guard": {k: after_results["no_repeat_guard"][k] for k in ("wer", "cer")},
}
with open(RESULTS_DIR / "finetuned_metrics.json", "w", encoding="utf-8") as f:
    json.dump(finetuned_metrics, f, indent=2)

if "trainer" in globals():
    with open(RESULTS_DIR / "training_log.json", "w", encoding="utf-8") as f:
        json.dump(trainer.state.log_history, f, indent=2)

!ls -lh {RESULTS_DIR}

NameError: name 'after_results' is not defined

In [ ]:
%cd /content/Nepali-Whisper-ASR
!git status --short
!git add results
!git commit -m "Add fine-tuned Whisper-small results and training log"
!git push

/content/Nepali-Whisper-ASR
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Everything up-to-date


In [ ]:
FINAL_MODEL_DIR = CHECKPOINT_DIR.parent / "final_model"
if FINAL_MODEL_DIR.exists():
    model_dir = FINAL_MODEL_DIR
else:
    model_dir = sorted(CHECKPOINT_DIR.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[1]))[-1]
print("loading:", model_dir)

model = WhisperForConditionalGeneration.from_pretrained(model_dir).to(device)
_ = model.eval()

loading: /content/drive/MyDrive/nepali_whisper_asr/final_model


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

In [ ]:
!ls -lh /content/Nepali-Whisper-ASR/results
!cat /content/Nepali-Whisper-ASR/results/finetuned_metrics.json
!git -C /content/Nepali-Whisper-ASR log --oneline -5

total 288K
-rw-r--r-- 1 root root  167 Nov  5 01:38 baseline_metrics.json
-rw-r--r-- 1 root root 112K Nov  5 01:38 baseline_predictions.csv
-rw-r--r-- 1 root root  306 Nov  5 01:38 finetuned_metrics.json
-rw-r--r-- 1 root root 150K Nov  5 01:38 finetuned_predictions.csv
drwxr-xr-x 2 root root 4.0K Nov  5 01:38 splits
-rw-r--r-- 1 root root  12K Nov  5 01:38 training_log.json
{
  "model": "whisper-small fine-tuned on SLR54 shard 0",
  "train_clips": 7961,
  "epochs": 3,
  "learning_rate": 1e-05,
  "effective_batch_size": 16,
  "test_clips": 898,
  "with_repeat_guard": {
    "wer": 0.391,
    "cer": 0.106
  },
  "without_repeat_guard": {
    "wer": 0.382,
    "cer": 0.104
  }
}d73b826 (HEAD -> main, origin/main, origin/HEAD) Add fine-tuned Whisper-small results and training log
24428f1 Add baseline Whisper-small results and data splits
884259b Add speaker-based train/val/test split for asr_nepali_0
21f2266 Initial project structure
64c63ff Initial commit


## Results

Test set: 898 clips from speakers never seen in training. No test text appears in the training set.

| Model | Generation settings | WER | CER |
|---|---|---|---|
| whisper-small (no fine-tuning) | repeat guard | 1.363 | 0.589 |
| whisper-small (fine-tuned) | repeat guard | 0.391 | 0.106 |
| whisper-small (fine-tuned) | no repeat guard | 0.382 | 0.104 |

WER can be above 1 because inserted words count as errors. The untrained model often repeats words,
so CER is easier to read here.

## Training setup
- Data: OpenSLR SLR54, one shard (9,935 of 157,905 utterances), split by speaker 80/10/10
- Train 7,961 / validation 933 / test 898 clips
- 3 epochs, learning rate 1e-5, effective batch size 16, fp16, T4 GPU (about 72 minutes)
- Validation CER (300 clips) went from 0.177 after half an epoch to 0.109 after 3 epochs

## Limits
- Training data is mostly short read phrases (about 3 words). The model may be weaker on long, natural speech.
- Only one shard of the dataset was used.
- Validation loss stopped improving after epoch 2 while training loss kept falling, so the model started to memorize.

## Error analysis

In [ ]:
results = pd.read_csv(REPO_DIR / "results" / "finetuned_predictions.csv")
results = results.fillna("")

PRED_COLUMN = "prediction_no_guard"


def calculate_cer(reference, prediction):
    try:
        return jiwer.cer(reference, prediction)
    except ValueError:
        return 1.0


# Calculate CER and check exact matches
results["cer"] = [
    calculate_cer(ref, pred)
    for ref, pred in zip(results["text"], results[PRED_COLUMN])
]

results["is_exact"] = results["text"] == results[PRED_COLUMN]

# Group clips based on the number of words
results["n_words"] = results["text"].str.split().str.len()
results["n_words"] = results["n_words"].clip(upper=4)

# Check whether the reference text contains digits
results["has_digit"] = results["text"].str.contains(r"[०-९0-9]")


# Overall results
print("Total clips:", len(results))
print("Exact match:", round(results["is_exact"].mean() * 100, 1), "%")


# CER based on the number of words
print("\nCER by number of words:")
print(results.groupby("n_words")["cer"].agg(["count", "mean"]).round(3))


# CER for clips with and without digits
print("\nCER for clips with and without digits:")
print(results.groupby("has_digit")["cer"].agg(["count", "mean"]).round(3))


# Calculate word-level errors
word_output = jiwer.process_words(
    results["text"].tolist(),
    results[PRED_COLUMN].tolist()
)

print("\nWord-level results:")
print("Hits:", word_output.hits)
print("Substitutions:", word_output.substitutions)
print("Deletions:", word_output.deletions)
print("Insertions:", word_output.insertions)


# Display the 10 clips with the highest CER
print("\n10 clips with the highest CER:")

worst_results = results.sort_values(by="cer", ascending=False).head(10)

for _, row in worst_results.iterrows():
    print("Reference:", row["text"])
    print("Prediction:", row[PRED_COLUMN])
    print()


# Display 15 randomly selected incorrect predictions
print("15 randomly selected incorrect clips:")

wrong_results = results[results["is_exact"] == False]

for _, row in wrong_results.sample(15, random_state=SEED).iterrows():
    print("Reference:", row["text"])
    print("Prediction:", row[PRED_COLUMN])
    print()

Total clips: 898
Exact match: 32.5 %

CER by number of words:
         count   mean
n_words              
1            2  0.174
2          169  0.100
3          661  0.097
4           66  0.137

CER for clips with and without digits:
           count   mean
has_digit              
False        834  0.095
True          64  0.183

Word-level results:
Hits: 1705
Substitutions: 846
Deletions: 86
Insertions: 75

10 clips with the highest CER:
Reference: १७०००० सेना रहेको देखियो
Prediction: १७ सुन्य शुन्य शुन्य सेनारहेको देख्यो

Reference: ६८ को बजेटमा
Prediction: अर्षट्रिकको बजेटमा

Reference: ऐन २०६४
Prediction: एनु २००६

Reference: २००८ मा समाप्त
Prediction: द्योजय आठमा समाप्त

Reference: ऋषिकेषका लक्ष्‍मण पिङको
Prediction: रेशी केसका लाँचमाड पिङको

Reference: खान अनुभव र आँट
Prediction: खानाउनु भब्र ८

Reference: चार बल प्रतियोगितामा
Prediction: ४ बल्कतीय दृतामा

Reference: प्रह ७
Prediction: प्रह साथ

Reference: उत्तर र दक्षिण
Prediction: उत्तर ढछिन्

Reference: तापक्रम ४२ डिग्री
Predic

## 6. Error analysis on the test set

Goal: find out *why* the fine-tuned model still makes mistakes.
The questions I want to answer:
1. How many "wrong" clips are only wrong because of punctuation or spaces?
2. Does the model write digits where the reference has words, or the other way round?
3. How much do WER and CER improve if I ignore punctuation?
4. Which single characters does the model mix up most often?

I use the predictions *without* the repeat guard, because they were slightly better.

In [ ]:
from collections import Counter
from difflib import SequenceMatcher

import jiwer
import pandas as pd

PREDICTION_COLUMNS= "prediction_no_guard"

results=pd.read_csv(REPO_DIR / "results" / "finetuned_predictions.csv")
results=results.fillna("") #an empty prediction is read as Nan, so turn it back into ""


results["reference_text"]=results["text"]
results["predicted_text"]=results[PREDICTION_COLUMNS]
results["is_wrong"]=results["reference_text"]!=results["predicted_text"]

print("total clips:",len(results))
print("wrong clips:",results["is_wrong"].sum())

total clips: 898
wrong clips: 606


### Question 1: punctuation and spaces
I remove punctuation and extra spaces from both texts.
If the texts are then equal, the clip was only wrong because of punctuation.

In [ ]:
PUNCTUATION_MARKS = ["।", ",", ".", "?", "!", ";", ":", '"', "'", "(", ")", "-", "–", "—"]


def remove_punctuation(text):
    for mark in PUNCTUATION_MARKS:
        text = text.replace(mark, "")
    return text


def normalize_text(text):
    text = remove_punctuation(text)
    return " ".join(text.split())   # one single space between words


results["reference_clean"] = results["reference_text"].apply(normalize_text)
results["prediction_clean"] = results["predicted_text"].apply(normalize_text)

# wrong in the raw text, but equal after removing punctuation
only_punctuation_wrong = results["is_wrong"] & (results["reference_clean"] == results["prediction_clean"])

# different after cleaning, but equal if I also remove ALL spaces (joined or split words)
reference_no_spaces = results["reference_clean"].str.replace(" ", "")
prediction_no_spaces = results["prediction_clean"].str.replace(" ", "")
only_spaces_wrong = (results["reference_clean"] != results["prediction_clean"]) & (reference_no_spaces == prediction_no_spaces)

print("wrong clips:", results["is_wrong"].sum())
print("wrong only because of punctuation:", only_punctuation_wrong.sum())
print("wrong only because of spaces (joined or split words):", only_spaces_wrong.sum())

wrong clips: 606
wrong only because of punctuation: 2
wrong only because of spaces (joined or split words): 30


### Question 2: digits
Some references use digits (like ४२) and some use words (like चार).
I count the clips where one side has digits and the other side does not.

In [ ]:
DIGIT_CHARACTERS = list("०१२३४५६७८९0123456789")


def contains_digit(text):
    for character in text:
        if character in DIGIT_CHARACTERS:
            return True
    return False


results["reference_has_digit"] = results["reference_text"].apply(contains_digit)
results["prediction_has_digit"] = results["predicted_text"].apply(contains_digit)

reference_only = results["reference_has_digit"] & ~results["prediction_has_digit"]
prediction_only = ~results["reference_has_digit"] & results["prediction_has_digit"]

print("reference has digits, prediction has none:", reference_only.sum())
print("prediction has digits, reference has none:", prediction_only.sum())

reference has digits, prediction has none: 14
prediction has digits, reference has none: 6


### Question 3: raw vs normalized scores
Raw WER and CER stay the main numbers.
The normalized numbers (punctuation removed) are only an extra, to show how much of the error is formatting.

In [ ]:
raw_wer = jiwer.wer(results["reference_text"].tolist(), results["predicted_text"].tolist())
raw_cer = jiwer.cer(results["reference_text"].tolist(), results["predicted_text"].tolist())

normalized_wer = jiwer.wer(results["reference_clean"].tolist(), results["prediction_clean"].tolist())
normalized_cer = jiwer.cer(results["reference_clean"].tolist(), results["prediction_clean"].tolist())

print(f"raw        -> WER {raw_wer:.3f} | CER {raw_cer:.3f}")
print(f"normalized -> WER {normalized_wer:.3f} | CER {normalized_cer:.3f}  (punctuation removed)")

raw        -> WER 0.382 | CER 0.104
normalized -> WER 0.381 | CER 0.103  (punctuation removed)


### Question 4: which characters are mixed up?
I compare each reference with its prediction, character by character.
When exactly one character was replaced by exactly one other character, I count that pair.
Note: a "character" here is one Unicode piece, so a vowel sign like ि is counted on its own.

In [ ]:
mixup_counter = Counter()

for reference_text, predicted_text in zip(results["reference_clean"], results["prediction_clean"]):
    matcher = SequenceMatcher(None, reference_text, predicted_text, autojunk=False)

    for operation, ref_start, ref_end, pred_start, pred_end in matcher.get_opcodes():
        one_character_replaced = (
            operation == "replace"
            and ref_end - ref_start == 1
            and pred_end - pred_start == 1
        )
        if one_character_replaced:
            wrong_pair = (reference_text[ref_start], predicted_text[pred_start])
            mixup_counter[wrong_pair] += 1

print("most common single-character mix-ups (reference -> prediction):")
for (reference_char, predicted_char), count in mixup_counter.most_common(15):
    print(f"  {reference_char}  ->  {predicted_char}   {count} times")

most common single-character mix-ups (reference -> prediction):
  ी  ->  ि   30 times
  श  ->  स   22 times
  ि  ->  ी   21 times
  े  ->  ि   18 times
  स  ->  श   16 times
  ब  ->  व   12 times
  ू  ->  ु   9 times
  व  ->  ब   8 times
  ग  ->  क   8 times
  ो  ->  ु   8 times
  द  ->  ध   8 times
  ध  ->  द   8 times
  ु  ->  ू   7 times
  ि  ->  े   6 times
  थ  ->  त   6 times


### Question 5: spelling problem or hearing problem?
I merge letters that I think sound the same: ी and ि, ू and ु, श/ष/स, ब and व.
If the reference and the prediction become equal after merging, the model heard the word
correctly and only chose a different spelling.

This is a diagnosis, not a better model. The raw WER and CER stay the main numbers.

In [ ]:
# Each letter on the left is replaced by the letter on the right.
# I chose these groups from the mix-up table. ष was not in the table: it is my own addition,
# so delete that line if you do not want it.
SPELLING_MERGE_MAP = {
    "ी": "ि",
    "ू": "ु",
    "श": "स",
    "ष": "स",
    "व": "ब",
}


def merge_confusable_letters(text):
    for original_letter, merged_letter in SPELLING_MERGE_MAP.items():
        text = text.replace(original_letter, merged_letter)
    return text


reference_merged = results["reference_clean"].apply(merge_confusable_letters)
prediction_merged = results["prediction_clean"].apply(merge_confusable_letters)

merged_wer = jiwer.wer(reference_merged.tolist(), prediction_merged.tolist())
merged_cer = jiwer.cer(reference_merged.tolist(), prediction_merged.tolist())

exact_before = (results["reference_clean"] == results["prediction_clean"]).sum()
exact_after = (reference_merged == prediction_merged).sum()

print(f"raw                -> WER {raw_wer:.3f} | CER {raw_cer:.3f}")
print(f"punctuation removed -> WER {normalized_wer:.3f} | CER {normalized_cer:.3f}")
print(f"letters merged      -> WER {merged_wer:.3f} | CER {merged_cer:.3f}")
print()
print("exactly correct clips before merging:", exact_before)
print("exactly correct clips after merging: ", exact_after)

raw                -> WER 0.382 | CER 0.104
punctuation removed -> WER 0.381 | CER 0.103
letters merged      -> WER 0.352 | CER 0.092

exactly correct clips before merging: 294
exactly correct clips after merging:  333


In [ ]:
fixed_by_merging = (
    (results["reference_clean"] != results["prediction_clean"])
    & (reference_merged == prediction_merged)
)
print("clips that become correct only after merging:", fixed_by_merging.sum())
print()

for row_index in results[fixed_by_merging].head(10).index:
    print("REF :", results.loc[row_index, "reference_clean"])
    print("PRED:", results.loc[row_index, "predicted_text"])
    print()

clips that become correct only after merging: 39

REF : अन्यथा मिल्दोजुल्दो शीर्षक
PRED: अन्यथा मिल्दोजुल्दो शिर्षक

REF : अन्वेषण गरे
PRED: अन्वेशण गरे

REF : अमृतसरी काम गर्छ
PRED: अमृतशरी काम गर्छ

REF : आदिवासी जनजाति महिला
PRED: आदिवासी जनजाती महिला

REF : ईश्वर बरालको विवाह
PRED: ईश्वर वरालको विवाह

REF : काम गर्ने अनुशासन
PRED: काम गर्ने अनुसासन

REF : कोइँच कोच कुकी
PRED: कोइँच कोच कुकि

REF : तीन पटक प्रथम
PRED: तिन पटक प्रथम

REF : तीनै काण्ड कण्ठस्थ
PRED: तिनै काण्ड कण्ठस्थ

REF : त्यतिबेला रक्त शुद्धताको
PRED: त्यतिबेला रक्त सुद्धताको



## Error analysis (898 test clips, no repeat guard)
- 606 of 898 clips are not an exact match (exact match 32.5%).
- Word errors: 846 substitutions, 86 deletions, 75 insertions.
- Punctuation is almost irrelevant: raw WER/CER 0.382/0.104 vs 0.381/0.103 with punctuation removed.
  Only 2 clips differ by punctuation alone, and 30 clips differ only by joined or split words.
- Digits: CER 0.183 on the 64 clips with digits in the reference vs 0.095 on the 834 without.
  In 14 of the 64 clips the model wrote no digits.
- The most common single-character mix-ups are vowel length (ी/ि, ू/ु), श/स, ब/व and द/ध.
  Many of these look like spelling choices rather than mishearing (to be confirmed, see next section).

### Spelling-tolerant diagnostic (not a model score)
Letters that I believe sound the same (ी/ि, ू/ु, श/ष/स, ब/व) were merged in both the reference
and the prediction before scoring.
- WER 0.352 and CER 0.092 (raw: 0.382 and 0.104)
- 39 clips became exactly correct (333 vs 294 of 898)
- Only about 6% of the wrong clips are wrong purely because of these spelling groups,
  so most of the remaining errors have other causes.

### Question 6: what are the remaining errors?
565 clips are still wrong after merging the confusable letters.
I take a random sample of 50, save them in a csv file, and label each one by hand.
Categories I will use:
- spelling_other: same word, different spelling, not covered by the merge
- word_boundary: words joined or split
- number: digit written as a word, or the other way round, or a wrong number
- name_or_rare_word: names and rare words
- mishearing: a different word or a different sound
- other

In [ ]:
SAMPLE_SIZE = 50

still_wrong = reference_merged != prediction_merged
print("clips still wrong after merging:", still_wrong.sum())   # I expect 565 (898 - 333)

review_table = results.loc[still_wrong, ["file_id", "speaker_id", "reference_clean", "prediction_clean"]]
review_table = review_table.sample(SAMPLE_SIZE, random_state=SEED)
review_table["category"] = ""   # fill this column by hand
review_table["notes"] = ""

review_path = REPO_DIR / "results" / "manual_error_review.csv"
review_table.to_csv(review_path, index=False, encoding="utf-8-sig")   # utf-8-sig so Excel shows Nepali correctly
print("saved:", review_path)

review_table.head(10)

clips still wrong after merging: 565
saved: /content/Nepali-Whisper-ASR/results/manual_error_review.csv


,file_id,speaker_id,reference_clean,prediction_clean,category,notes
858,06debe66b6,885af,सेयर गर्न सक्थेँ,सेर गर्न सक्थे,,
823,05aa243a57,b365e,सम्म ८५ हजार,सम्म ५०० हजार,,
342,035cadbf56,12e6a,ठेगानाको नगर वा,ठेका नाको नगवरवा,,
824,07ad3a940c,bdfe3,सम्मको औसत १९३,सम्मको औषद १३३,,
765,00a63d1092,5b02a,वसन्त ऋतुको,वस्दन्त रितुको,,
481,0380e6eaeb,2fb12,पन्तले सानैदेखि मामलका,पन्तले सानेदेखि मामलका,,
358,03d7a3b6f9,11542,तापक्रम ४२ डिग्री,ताप्रम बाइलेस डिक्री,,
783,0543d5ce1e,90913,व्यक्तिहरूलाई भूमिमा राख्ने अर्थी,व्यक्तिहरू उला भूमीमा राख्ने अर्थि,,
601,053388cb25,96e21,भाषाको नाटक हो,भाषाकको नाटक हो,,
324,02b148d559,61821,जीवनमा अस्तित्वको प्रश्न,जीवतमा अस्तित्तको प्रश्न,,


In [ ]:
from IPython.display import Audio


def play_clip(file_id):
    audio_path = test_df.loc[test_df["file_id"] == file_id, "audio_path"].iloc[0]
    waveform, sample_rate = sf.read(audio_path)
    return Audio(waveform, rate=sample_rate)


# example: play_clip("PUT_A_FILE_ID_FROM_THE_TABLE_HERE")

In [8]:
# ===== CELL 1: audio and repo (run first, every new session) =====
import shutil
import zipfile
from pathlib import Path

from google.colab import drive, userdata

GITHUB_USER = "Susan-sapkota-98"
DOWNLOAD_URL = "https://openslr.trmal.net/resources/54/asr_nepali_0.zip"
MINIMUM_ZIP_SIZE_MB = 500
EXPECTED_AUDIO_FILES = 9935

drive.mount("/content/drive")
PROJECT_DRIVE_DIR = Path("/content/drive/MyDrive/nepali_whisper_asr")
DRIVE_ZIP_PATH = PROJECT_DRIVE_DIR / "data" / "asr_nepali_0.zip"

LOCAL_DATA_DIR = Path("/content/data")
LOCAL_ZIP_PATH = LOCAL_DATA_DIR / "asr_nepali_0.zip"
AUDIO_DIR = LOCAL_DATA_DIR / "audio"
REPO_DIR = Path("/content/Nepali-Whisper-ASR")


def zip_is_healthy(zip_path):
    """A good zip is big enough and has a readable zip structure."""
    if not zip_path.exists():
        return False
    size_in_mb = zip_path.stat().st_size / 1024 / 1024
    return size_in_mb > MINIMUM_ZIP_SIZE_MB and zipfile.is_zipfile(zip_path)


def count_audio_files():
    return len(list(AUDIO_DIR.rglob("*.flac")))


LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

if count_audio_files() != EXPECTED_AUDIO_FILES:
    if AUDIO_DIR.exists():
        shutil.rmtree(AUDIO_DIR)

    if not zip_is_healthy(LOCAL_ZIP_PATH):
        if zip_is_healthy(DRIVE_ZIP_PATH):
            shutil.copy(DRIVE_ZIP_PATH, LOCAL_ZIP_PATH)
            print("zip copied from Drive")
        else:
            print("downloading the zip from OpenSLR (a few minutes) ...")
            !wget -q -O {LOCAL_ZIP_PATH} {DOWNLOAD_URL}
            if zip_is_healthy(LOCAL_ZIP_PATH):
                DRIVE_ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy(LOCAL_ZIP_PATH, DRIVE_ZIP_PATH)
                print("good zip saved to Drive")

    assert zip_is_healthy(LOCAL_ZIP_PATH), "the zip is broken or missing - send me the lines printed above"
    !unzip -q {LOCAL_ZIP_PATH} -d {AUDIO_DIR}

audio_file_count = count_audio_files()
assert audio_file_count == EXPECTED_AUDIO_FILES, f"audio files: {audio_file_count}, expected {EXPECTED_AUDIO_FILES}"
print("audio files:", audio_file_count)

if not (REPO_DIR / ".git").exists():
    if REPO_DIR.exists():
        REPO_DIR.rename(f"{REPO_DIR}_leftover")
    github_token = userdata.get("GITHUB_TOKEN")
    !git clone -q https://{github_token}@github.com/{GITHUB_USER}/Nepali-Whisper-ASR.git {REPO_DIR}

print("repo ready:", (REPO_DIR / ".git").exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
audio files: 9935
repo ready: True


In [ ]:
# ===== LABELING CELL v2 (buttons): label the 50 wrong clips by hand =====
import ipywidgets as widgets
import pandas as pd
import soundfile as sf
from IPython.display import Audio, display

for needed_name in ["AUDIO_DIR", "REPO_DIR", "PROJECT_DRIVE_DIR"]:
    assert needed_name in globals(), f"{needed_name} is missing - run Cell 1 (audio and repo) first"

# button text, and the category name that is saved in the file
CATEGORY_BUTTONS = [
    ("1  spelling_other", "spelling_other"),
    ("2  word_boundary", "word_boundary"),
    ("3  number", "number"),
    ("4  name_or_rare_word", "name_or_rare_word"),
    ("5  mishearing", "mishearing"),
    ("6  other", "other"),
    ("7  reference_wrong", "reference_wrong"),
]

REVIEW_FILE_IN_REPO = REPO_DIR / "results" / "manual_error_review.csv"
LABELED_FILE_ON_DRIVE = PROJECT_DRIVE_DIR / "manual_error_review_labeled_v2.csv"

# start from the empty table in the repo, or continue from the saved v2 file
review_path = LABELED_FILE_ON_DRIVE if LABELED_FILE_ON_DRIVE.exists() else REVIEW_FILE_IN_REPO
review_table = pd.read_csv(review_path, encoding="utf-8-sig", dtype=str).fillna("")
if review_path == REVIEW_FILE_IN_REPO:
    review_table["category"] = ""   # make sure we start clean
review_table.to_csv(LABELED_FILE_ON_DRIVE, index=False, encoding="utf-8-sig")
print("answers are saved to:", LABELED_FILE_ON_DRIVE)

audio_path_by_id = {path.stem: str(path) for path in AUDIO_DIR.rglob("*.flac")}
rows_to_label = review_table.index[review_table["category"] == ""].tolist()
print("clips left to label:", len(rows_to_label))

state = {"position": 0}
screen = widgets.Output()


def save_answer(row_index, category_name):
    review_table.loc[row_index, "category"] = category_name
    review_table.to_csv(LABELED_FILE_ON_DRIVE, index=False, encoding="utf-8-sig")


def make_category_button(button_text, category_name, row_index):
    button = widgets.Button(description=button_text, layout=widgets.Layout(width="200px"))

    def on_click(clicked_button):
        save_answer(row_index, category_name)
        state["position"] += 1
        show_current_clip()

    button.on_click(on_click)
    return button


def go_back(clicked_button):
    state["position"] = max(0, state["position"] - 1)
    show_current_clip()


def show_current_clip():
    screen.clear_output(wait=True)
    with screen:
        if state["position"] >= len(rows_to_label):
            print("all clips in this list are labeled")
            print(review_table["category"].replace("", "not labeled yet").value_counts())
            return

        row_index = rows_to_label[state["position"]]
        row = review_table.loc[row_index]
        waveform, sample_rate = sf.read(audio_path_by_id[row["file_id"]])

        print(f"clip {state['position'] + 1} of {len(rows_to_label)}")
        print("REF :", row["reference_clean"])
        print("PRED:", row["prediction_clean"])
        display(Audio(waveform, rate=sample_rate))
        print("Step 1: does the audio match PRED and not REF?  -> click 7")
        print("Step 2: otherwise, what is the difference between REF and PRED?  -> click 1 to 6")

        category_buttons = [make_category_button(text, name, row_index) for text, name in CATEGORY_BUTTONS]
        back_button = widgets.Button(description="back (undo)", layout=widgets.Layout(width="200px"))
        back_button.on_click(go_back)
        display(widgets.HBox(category_buttons, layout=widgets.Layout(flex_flow="row wrap")))
        display(back_button)


display(screen)
show_current_clip()

all clips in this list are labeled
category
spelling_other       26
mishearing           10
word_boundary         6
reference_wrong       4
number                3
name_or_rare_word     1
Name: count, dtype: int64


In [ ]:
# import json
# import re
# import subprocess

# NOTEBOOK_PATH = REPO_DIR / "notebooks" / "01_whisper_nepali_finetune.ipynb"
# TOKEN_PATTERN = re.compile(r"(github_pat_|ghp_)[A-Za-z0-9_]{10,}")
# FORBIDDEN_SUFFIXES = (".zip", ".flac", ".wav", ".mp3", ".safetensors", ".pt", ".bin")

# GITIGNORE_LINES = [
#     "# data and audio",
#     "*.zip", "*.flac", "*.wav", "*.mp3", "data/",
#     "",
#     "# models and training output",
#     "checkpoints/", "final_model/", "smoke_test/", "*.safetensors", "*.pt", "*.bin",
#     "",
#     "# python and notebook leftovers",
#     "__pycache__/", ".ipynb_checkpoints/",
# ]


# def run_git(*arguments):
#     completed = subprocess.run(
#         ["git", "-C", str(REPO_DIR), *arguments],
#         capture_output=True, text=True,
#     )
#     return completed.stdout + completed.stderr


# # 1. remove any real token from the copy of the notebook (the original in Drive is not touched)
# assert NOTEBOOK_PATH.exists(), "notebook copy not found in the repo"
# notebook_text = NOTEBOOK_PATH.read_text(encoding="utf-8")
# cleaned_text, removed_count = TOKEN_PATTERN.subn("[REMOVED_TOKEN]", notebook_text)
# print("tokens removed:", removed_count)

# cleaned_notebook = json.loads(cleaned_text)
# cleaned_notebook.get("metadata", {}).pop("widgets", None)   # old widget data can break the GitHub preview
# NOTEBOOK_PATH.write_text(json.dumps(cleaned_notebook, ensure_ascii=False, indent=1), encoding="utf-8")

# # 2. final safety check
# assert not TOKEN_PATTERN.search(NOTEBOOK_PATH.read_text(encoding="utf-8")), "token still in notebook - do not push"

# # 3. .gitignore
# (REPO_DIR / ".gitignore").write_text("\n".join(GITIGNORE_LINES) + "\n", encoding="utf-8")

# # 4. stage only the folders we want
# run_git("add", ".gitignore", "notebooks", "results")

# staged_files = run_git("diff", "--cached", "--name-only").splitlines()
# bad_files = [name for name in staged_files if name.endswith(FORBIDDEN_SUFFIXES) or name.startswith("data/")]
# assert not bad_files, f"stop, these files must not be pushed: {bad_files}"
# print("files to push:")
# for name in staged_files:
#     print("  ", name)

# # 5. commit and push
# print(run_git("commit", "-m", "Add notebook and results"))
# print(run_git("push"))

tokens removed: 0
files to push:
   results/splits/test.csv
   results/splits/train.csv
   results/splits/val.csv
[main c297d1f] Add notebook and results
 3 files changed, 9792 deletions(-)

To https://github.com/Susan-sapkota-98/Nepali-Whisper-ASR.git
   5486669..c297d1f  main -> main



In [9]:
# ===== SUMMARY CELL: what are the remaining errors? =====
import pandas as pd

LABELED_FILE = PROJECT_DRIVE_DIR / "manual_error_review_labeled_v2.csv"
STILL_WRONG_TOTAL = 565   # clips that were still wrong after merging the letters

labeled_table = pd.read_csv(LABELED_FILE, encoding="utf-8-sig", dtype=str).fillna("")
labeled_table = labeled_table[labeled_table["category"] != ""]
sample_size = len(labeled_table)

category_counts = labeled_table["category"].value_counts()
category_share = category_counts / sample_size
margin_of_error = 1.96 * (category_share * (1 - category_share) / sample_size) ** 0.5   # rough, small sample

summary_table = pd.DataFrame({
    "clips_in_sample": category_counts,
    "percent": (category_share * 100).round(1),
    "plus_minus": (margin_of_error * 100).round(1),
    "estimated_of_565": (category_share * STILL_WRONG_TOTAL).round(0).astype(int),
})
print("labeled clips:", sample_size)
print(summary_table)

summary_table.to_csv(PROJECT_DRIVE_DIR / "manual_error_summary.csv", encoding="utf-8-sig")

labeled clips: 50
                   clips_in_sample  percent  plus_minus  estimated_of_565
category                                                                 
spelling_other                  26     52.0        13.8               294
mishearing                      10     20.0        11.1               113
word_boundary                    6     12.0         9.0                68
reference_wrong                  4      8.0         7.5                45
number                           3      6.0         6.6                34
name_or_rare_word                1      2.0         3.9                11
